# 05 — Power Flow Projections (Scaffold)

**Purpose:** Extend the BA interchange graph into a projection tool by adding
time-series dimensions: historical trends, seasonal patterns, and scenario-based
capacity change projections.

This notebook is a scaffold — it defines the analytical structure and data
requirements for the projection layer without fully implementing them.
Complete notebooks 03a, 03b, and 04 first.

**The projection framing:**
The energy metabolism map has three temporal layers:
1. **Historical baseline** — what flows existed and how they changed (EIA-930 multi-year)
2. **Current snapshot** — the most recent year's flow graph (from 03b)
3. **Projected scenarios** — how the graph changes under capacity addition or retirement

**Inputs:**
- `data/processed/eia930_raw.parquet` — from 03b (extend to multi-year in this notebook)
- `data/processed/grid_flow_network.graphml` — from 03b
- `data/processed/ba_capacity_summary.csv` — from 03a

**Outputs:**
- `data/processed/eia930_multiyear.parquet` — 2018-2023 hourly interchange
- `data/processed/ba_flow_trends.csv` — annual BA-pair flow aggregates by year
- `data/processed/projection_scenarios.csv` — scenario capacity deltas
- `data/processed/projected_flow_map.html` — interactive scenario map

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import geopandas as gpd
import networkx as nx
import utils

## Step 1 — Extend EIA-930 to Multi-Year

Re-run the paginated EIA-930 fetch from 03b for each year in the range
2018-2023. Concatenate into a single parquet file partitioned by year.

Implementation notes:
- Loop `START_DATE` / `END_DATE` over years
- Reuse `utils.eia_get()` and pagination logic from 03b exactly
- Append each year to a list, then `pd.concat()` and save as parquet
- Total size: ~1.5M rows — manageable in memory

In [2]:
# ── Multi-year fetch using EIA Grid Monitor bulk CSVs ─────────────────────────
# Same approach as 03b: six-month bulk CSVs, cached locally.
import requests

YEARS = list(range(2018, 2024))  # 2018 through 2023
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'eia930'
RAW_DIR.mkdir(parents=True, exist_ok=True)

MULTIYEAR_PATH = PROJECT_ROOT / 'data' / 'processed' / 'eia930_multiyear.parquet'

if MULTIYEAR_PATH.exists():
    print(f'Multi-year parquet already exists: {MULTIYEAR_PATH}')
    df_multi = pd.read_parquet(MULTIYEAR_PATH)
    print(f'Loaded: {len(df_multi):,} rows, years {df_multi["year"].unique().tolist()}')
else:
    base_url = (
        'https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/'
        'EIA930_INTERCHANGE_{year}_{half}.csv'
    )
    col_map = {
        'Balancing Authority':                         'fromba',
        'Directly Interconnected Balancing Authority': 'toba',
        'Interchange (MW)':                            'value',
        'UTC Time at End of Hour':                     'period',
    }

    frames = []
    for year in YEARS:
        for half in ['Jan_Jun', 'Jul_Dec']:
            fname = RAW_DIR / f'EIA930_INTERCHANGE_{year}_{half}.csv'
            if not fname.exists():
                url = base_url.format(year=year, half=half)
                print(f'Downloading {fname.name}...', end=' ', flush=True)
                r = requests.get(url, timeout=300, stream=True)
                r.raise_for_status()
                with open(fname, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                print(f'{fname.stat().st_size / 1024 / 1024:.1f} MB')
            else:
                print(f'Cached: {fname.name}')

            df = pd.read_csv(fname, usecols=list(col_map.keys()))
            df = df.rename(columns=col_map)
            df['period'] = pd.to_datetime(df['period'], utc=True, errors='coerce')
            df['value']  = pd.to_numeric(df['value'], errors='coerce')
            df['year']   = year
            frames.append(df)

    df_multi = pd.concat(frames, ignore_index=True)
    df_multi.to_parquet(MULTIYEAR_PATH, index=False)
    print(f'\nSaved multi-year parquet → {MULTIYEAR_PATH}')
    print(f'Total rows: {len(df_multi):,}')

101.2 MB


102.7 MB


100.4 MB


102.5 MB


99.5 MB


...

98.2 MB


98.4 MB


Cached: EIA930_INTERCHANGE_2023_Jan_Jun.csv


Cached: EIA930_INTERCHANGE_2023_Jul_Dec.csv



Saved multi-year parquet → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/eia930_multiyear.parquet
Total rows: 15,946,246


## Step 2 — Annual Trend Analysis

For each year and each directed BA pair, compute:
- `total_mwh` — annual net interchange
- `mean_mw` — mean hourly load on the corridor
- `peak_mw` — 95th percentile hourly load (stress metric)

Then identify corridors where flow has grown or reversed direction
between 2018 and 2023 — these are signals of structural grid change
(e.g. retirement of coal in a BA, large solar/wind buildout).

Implementation notes:
- Load the multi-year parquet
- Extract `year` from `period`
- `groupby(['year', 'fromba', 'toba'])` with `.agg(total_mwh=..., mean_mw=..., peak_mw=...)`
- Join with `ba_capacity_summary.csv` to correlate flow change with capacity change

In [3]:
# ── Annual trend analysis ─────────────────────────────────────────────────────
trends = (
    df_multi
    .dropna(subset=['value'])
    .groupby(['year', 'fromba', 'toba'])['value']
    .agg(
        total_mwh='sum',
        mean_mw='mean',
        peak_mw=lambda x: x.quantile(0.95),
    )
    .reset_index()
)

# ── Flag corridors that grew, shrank, or reversed direction 2018 → 2023 ───────
base  = trends[trends['year'] == 2018][['fromba', 'toba', 'total_mwh']].rename(columns={'total_mwh': 'mwh_2018'})
final = trends[trends['year'] == 2023][['fromba', 'toba', 'total_mwh']].rename(columns={'total_mwh': 'mwh_2023'})
corridor_delta = base.merge(final, on=['fromba', 'toba'], how='inner')
corridor_delta['delta_mwh']    = corridor_delta['mwh_2023'] - corridor_delta['mwh_2018']
corridor_delta['pct_change']   = corridor_delta['delta_mwh'] / corridor_delta['mwh_2018'].abs().replace(0, float('nan')) * 100
corridor_delta['sign_flip']    = (corridor_delta['mwh_2018'].apply(lambda x: 1 if x >= 0 else -1) !=
                                   corridor_delta['mwh_2023'].apply(lambda x: 1 if x >= 0 else -1))

# Save
trends_path = PROJECT_ROOT / 'data' / 'processed' / 'ba_flow_trends.csv'
trends.to_csv(trends_path, index=False)
print(f'Saved → {trends_path}  ({len(trends):,} rows)')

print(f'\nUnique directed corridors with 2018 and 2023 data: {len(corridor_delta):,}')
print(f'Corridors that reversed direction 2018→2023: {corridor_delta["sign_flip"].sum()}')
print('\nTop 10 corridors by absolute growth (MWh):')
print(corridor_delta.nlargest(10, 'delta_mwh')[['fromba','toba','mwh_2018','mwh_2023','pct_change']].to_string(index=False))

Saved → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/ba_flow_trends.csv  (1,839 rows)

Unique directed corridors with 2018 and 2023 data: 290
Corridors that reversed direction 2018→2023: 64

Top 10 corridors by absolute growth (MWh):
fromba toba    mwh_2018   mwh_2023  pct_change
   PJM MISO -33450551.0 56567112.0  269.106667
   PJM NYIS  -6645518.0 13841487.0  308.283041
   SRP AZPS  10433687.0 30108668.0  188.571700
  CISO LDWP -21232098.0 -2579071.0   87.852962
  CISO BPAT -15180000.0 -1141151.0   92.482536
  NYIS  HQT -11765335.0 -2707610.0   76.986546
  BPAT BCHA    962846.0  9478010.0  884.374448
  NEVP CISO   -925137.0  7559381.0  917.109358
  LDWP BPAT -13805438.0 -6138554.0   55.535246
  PSCO WACM  -4336705.0  2387612.0  155.055901


## Step 3 — Scenario Capacity Deltas

Define projection scenarios as changes to BA-level installed capacity.
These are simple what-if perturbations, not full dispatch simulations.

Example scenarios to implement:
- **Coal retirement:** Remove all coal MW from BAs where coal > 30% of total mix
- **Solar build-out:** Add solar MW proportional to NREL solar resource score per BA
- **Wind corridor:** Double wind capacity in MISO, SPP, and PJM West BAs

For each scenario, estimate the first-order effect on BA interchange using
a simple energy balance:
- If a BA loses baseload capacity, its net import need increases proportionally
- Distribute the increased import across neighbouring BAs weighted by
  their historical export share to that BA

This is a heuristic, not a power-flow simulation, but it gives directionally
useful projections for visualization purposes.

Implementation notes:
- Load `ba_capacity_summary.csv` and `ba_interchange_summary.csv`
- Define a `scenarios` dict: `{scenario_name: {ba_code: delta_mw}}`
- For each scenario, scale the historical edge weights by the implied
  demand shift and output a new `flow_summary` DataFrame
- Save to `projection_scenarios.csv`

In [4]:
import numpy as np

# ── Load capacity and flow summaries ─────────────────────────────────────────
cap = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'ba_capacity_summary.csv')
flow = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'ba_interchange_summary.csv')

# Flatten capacity pivot: ba_code + per-fuel columns → cast to dict
cap = cap.reset_index() if 'ba_code' not in cap.columns else cap
# ba_capacity_summary has ba_code/ba_name as index (MultiIndex) from the pivot
if cap.columns[0] not in ('ba_code', 'ba_name'):
    cap.columns.name = None
    cap = cap.rename(columns={cap.columns[0]: 'ba_code', cap.columns[1]: 'ba_name'})

# ── Scenario 1: Coal retirement ────────────────────────────────────────────────
# BAs where coal capacity > 30 % of total — retire all coal MW
coal_col = next((c for c in cap.columns if 'coal' in c.lower() or c == 'Coal'), None)
total_col = 'total_mw'
if coal_col and total_col in cap.columns:
    coal_heavy = cap[cap[coal_col] / cap[total_col].replace(0, np.nan) > 0.30].copy()
    coal_retirement = {row['ba_code']: -row[coal_col] for _, row in coal_heavy.iterrows()}
else:
    coal_retirement = {}

# ── Scenario 2: Solar build-out ────────────────────────────────────────────────
# Sun Belt BAs (CISO, AZPS, NEVP, PACE, WACM, SRP, EPE, PNM, SOCO, FPC, FPL, SCEG)
# Add 5,000 MW solar to each
SOLAR_BAS = {'CISO', 'AZPS', 'NEVP', 'PACE', 'WACM', 'SRP', 'EPE', 'PNM',
             'SOCO', 'FPC', 'FPL', 'SCEG'}
solar_buildout = {ba: 5000 for ba in SOLAR_BAS if ba in cap['ba_code'].values}

# ── Scenario 3: Wind corridor ─────────────────────────────────────────────────
# Double wind in MISO, SPP, PJM West BAs
wind_col = next((c for c in cap.columns if 'wind' in c.lower() or c == 'Wind'), None)
WIND_BAS = {'MISO', 'SWPP', 'PJM'}
if wind_col:
    wind_data = cap[cap['ba_code'].isin(WIND_BAS)].set_index('ba_code')[wind_col]
    wind_corridor = {ba: float(mw) for ba, mw in wind_data.items() if mw > 0}
else:
    wind_corridor = {}

SCENARIOS = {
    'coal_retirement': coal_retirement,
    'solar_buildout':  solar_buildout,
    'wind_corridor':   wind_corridor,
}

# ── Apply scenarios: first-order energy balance heuristic ─────────────────────
# If BA gains MW → it exports more (or imports less) → scale outgoing edges up
# If BA loses MW → it imports more → scale incoming edges up
# Scale factor: delta_mw / total_mw, applied to corridor mean_mw
def apply_scenario(flow_df, capacity_df, delta_dict):
    df = flow_df.copy()
    for ba_code, delta_mw in delta_dict.items():
        row = capacity_df[capacity_df['ba_code'] == ba_code]
        if row.empty:
            continue
        total = float(row[total_col].iloc[0])
        if total <= 0:
            continue
        scale = delta_mw / total
        # Export edges from this BA
        mask_out = df['fromba'] == ba_code
        df.loc[mask_out, 'mean_mw']    *= (1 + scale)
        df.loc[mask_out, 'total_mwh']  *= (1 + scale)
        # Import edges (inversely — if BA gains capacity, it imports less)
        mask_in = df['toba'] == ba_code
        df.loc[mask_in, 'mean_mw']    *= (1 - scale)
        df.loc[mask_in, 'total_mwh']  *= (1 - scale)
    return df

scenario_rows = []
for scen_name, delta_dict in SCENARIOS.items():
    scen_flow = apply_scenario(flow, cap, delta_dict)
    scen_flow['scenario'] = scen_name
    scenario_rows.append(scen_flow)
scen_flow_all = pd.concat([flow.assign(scenario='baseline')] + scenario_rows, ignore_index=True)

scen_path = PROJECT_ROOT / 'data' / 'processed' / 'projection_scenarios.csv'
scen_flow_all.to_csv(scen_path, index=False)
print(f'Saved → {scen_path}')
for name, d in SCENARIOS.items():
    print(f'  {name}: {len(d)} BAs affected, total delta = {sum(d.values()):,.0f} MW')

Saved → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/projection_scenarios.csv
  coal_retirement: 4 BAs affected, total delta = -13,041 MW
  solar_buildout: 11 BAs affected, total delta = 55,000 MW
  wind_corridor: 3 BAs affected, total delta = 33,812 MW


## Step 4 — Projected Flow Map

Build an interactive Folium map that can toggle between:
- The baseline 2023 flow graph (from 03b)
- Each projection scenario

Show changes in edge width and BA fill color (net export position) between
baseline and scenario. Use Folium's `LayerControl` to toggle scenarios.

Implementation notes:
- For each scenario, build a separate FeatureGroup with the projected edges
- Add all groups to the map with `LayerControl`
- Colour edges by direction of change: thicker/orange = more export,
  thinner/blue = more import relative to baseline

In [5]:
import folium
import networkx as nx

# ── Load BA geometry and graph ────────────────────────────────────────────────
ba = gpd.read_file(PROJECT_ROOT / 'data' / 'processed' / 'ba_territories.geojson')
G  = nx.read_graphml(str(PROJECT_ROOT / 'data' / 'processed' / 'grid_flow_network.graphml'))

# Compute centroids for edge drawing
ba_proj = ba.to_crs(epsg=5070)
ba['centroid_lon'] = ba_proj.geometry.centroid.to_crs(epsg=4326).x
ba['centroid_lat'] = ba_proj.geometry.centroid.to_crs(epsg=4326).y
centroids = ba.set_index('ba_code')[['centroid_lat', 'centroid_lon']].to_dict('index')

# ── Build map with one FeatureGroup per scenario ──────────────────────────────
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

# BA territory outlines (always on)
folium.GeoJson(
    ba.__geo_interface__,
    name='BA Territories',
    style_function=lambda _: {'fillColor': 'transparent', 'color': '#4466aa',
                               'weight': 0.7, 'opacity': 0.5},
    tooltip=folium.GeoJsonTooltip(fields=['ba_code', 'ba_name'], aliases=['BA', 'Name'])
).add_to(m)

SCEN_COLORS = {
    'baseline':       '#888888',
    'coal_retirement':'#cc4400',
    'solar_buildout': '#ddaa00',
    'wind_corridor':  '#0066cc',
}

baseline_flow = scen_flow_all[scen_flow_all['scenario'] == 'baseline'].set_index(['fromba', 'toba'])

for scen_name in ['baseline', 'coal_retirement', 'solar_buildout', 'wind_corridor']:
    fg = folium.FeatureGroup(name=scen_name.replace('_', ' ').title(), show=(scen_name == 'baseline'))
    scen_df = scen_flow_all[scen_flow_all['scenario'] == scen_name]
    top = scen_df.nlargest(80, 'mean_mw')
    max_mw = top['mean_mw'].max() if len(top) else 1

    for _, row in top.iterrows():
        src = centroids.get(row['fromba'])
        dst = centroids.get(row['toba'])
        if not src or not dst:
            continue
        width = 1 + 5 * (abs(row['mean_mw']) / max_mw)

        # Show delta vs baseline with opacity
        key = (row['fromba'], row['toba'])
        if scen_name != 'baseline' and key in baseline_flow.index:
            base_mw = baseline_flow.loc[key, 'mean_mw']
            delta_pct = (row['mean_mw'] - base_mw) / (abs(base_mw) + 1e-9) * 100
            tooltip_txt = (f"{row['fromba']} → {row['toba']}: {row['mean_mw']:,.0f} MW "
                           f"({delta_pct:+.1f}% vs baseline)")
        else:
            tooltip_txt = f"{row['fromba']} → {row['toba']}: {row['mean_mw']:,.0f} MW avg"

        folium.PolyLine(
            locations=[[src['centroid_lat'], src['centroid_lon']],
                       [dst['centroid_lat'], dst['centroid_lon']]],
            color=SCEN_COLORS[scen_name],
            weight=width,
            opacity=0.65,
            tooltip=tooltip_txt,
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

map_path = PROJECT_ROOT / 'data' / 'processed' / 'projected_flow_map.html'
m.save(str(map_path))
print(f'Map saved → {map_path}')

Map saved → /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/projected_flow_map.html
